###  CELL0: Install Libraries

In [1]:
!pip install DrissionPage pandas openpyxl

### Cell 1: Imports & Configuration

In [6]:
import pandas as pd
import os, time, random
from DrissionPage import ChromiumPage
from tqdm.notebook import tqdm # 进度条（可以没有）

# 自动定位桌面路径
save_path = os.path.join(os.path.expanduser("~"), "Desktop", "退役军人事务部新闻.xlsx") # 要确保文件夹存在
base_url = "https://www.mva.gov.cn/xinwen/dfdt/"

print(f"Ready. The file will be saved to：{save_path}")

Ready. The file will be saved to：/Users/kathyixiyue/Desktop/退役军人事务部新闻.xlsx


#假设文件夹命名为123，定位在桌面\
folder_name = "123" \
dir_path = os.path.join(os.path.expanduser("~"), "Desktop", folder_name)\

#检查文件夹是否存在，不存在就创建一个\
if not os.path.exists(dir_path):\
    os.makedirs(dir_path)\
    print(f"The folder was not found and has been automatically created: {dir_path}")\
else:\
    print(f"The folder already exists and does not need to be created.")

#拼接最终的文件保存路径\
save_path = os.path.join(dir_path, "退役军人事务部新闻.xlsx")

### Cell 2: Launch Browser

In [3]:
try:
    page = ChromiumPage()
    print("Chrome has been successfully launched!")
    print(f"Current Page Title:{page.title}")
except Exception as e:
    print(f"Failed to start. Please close all Chrome windows and try again. Error message: {e}")

Chrome has been successfully launched!
Current Page Title:吉林延吉：开展“情暖老兵·共庆元宵”活动-地方动态-中华人民共和国退役军人事务部


### Cell 3: Collect Article Links

In [6]:
start_page = 1
end_page = 1 

news_list = []

print(f"Start to scan from {start_page} to {end_page} ...")

for i in range(start_page, end_page + 1):
    url = f"{base_url}index.html" if i == 1 else f"{base_url}index_{i-1}.html"
    page.get(url)
    
    # 获取页面所有 li
    items = page.eles('tag:li')
    
    for item in items:
        link_node = item.ele('tag:a', timeout=0.5)
        date_node = item.ele('tag:span', timeout=0.5) # 新闻通常带有日期 span
        
        if link_node:
            href = link_node.attr('href')
            # 过滤逻辑：
            # 1. 必须包含日期（导航栏标签通常没有日期 span）
            # 2. 链接必须以 .html 结尾
            if date_node and date_node.text.strip() and href.endswith('.html'):
                
                full_url = base_url + href.replace('./', '') if href.startswith('./') else href
                
                news_list.append({
                    '标题': link_node.text.strip(),
                    '链接': full_url,
                    '日期': date_node.text.strip(),
                    '正文': ""
                })
    
    print(f"Page {i} completed. Total number of links: {len(news_list)}")
    time.sleep(1)

print(f"\n Link extraction completed!Total {len(news_list)} items.")

开始扫描第 1 到 1 页...
Page 1 完成。有效条目: 15

 提取完成！共计 15 条有效新闻。


### Cell 4: Scrape Article Body Text

In [7]:
print("Start extracting the full text of the content ...")

for record in tqdm(news_list):
    # 如果已经抓过正文了则跳过
    if record['正文']: continue
    
    try:
        page.get(record['链接'])
        # 核心逻辑：先找专用容器，找不到就抓所有 P 标签
        content_box = page.ele('.content') or page.ele('.TRS_Editor') or page.ele('#article_content')
        
        if content_box:
            text = content_box.text.strip()
        else:
            # 保底：抓取页面所有段落
            paras = page.eles('tag:p')
            text = "\n".join([p.text.strip() for p in paras if len(p.text.strip()) > 5])
        
        # 清洗：去掉落款
        for keyword in ["主办单位", "版权所有", "地址："]:
            if keyword in text:
                text = text.split(keyword)[0].strip()
        
        record['正文'] = text
        
    except Exception as e:
        record['正文'] = f"Failed to capture: {e}"
    
    # 动态保存：每抓 10 篇存一次，防止中断
    if (news_list.index(record) + 1) % 10 == 0:
        pd.DataFrame(news_list).to_excel(save_path, index=False)

# 最终保存
df = pd.DataFrame(news_list)
df.to_excel(save_path, index=False)

print(f"The file has been saved to the desktop.")
display(df.head()) # 在单元格下方显示结果

Start extracting the full text of the content ...


  0%|          | 0/15 [00:00<?, ?it/s]

The file has been saved to the desktop.


,标题,链接,日期,正文
0,山东济宁：书画迎新春 翰墨敬军魂,https://www.mva.gov.cn/xinwen/dfdt/202603/t202...,2026-03-09,山东济宁：书画迎新春 翰墨敬军魂\n为深切表达对退役老兵的尊崇与祝福，传承红色基因、赓续军人...
1,广东：召开人工智能与退役军人医疗创新应用座谈会,https://www.mva.gov.cn/xinwen/dfdt/202603/t202...,2026-03-09,广东：召开人工智能与退役军人医疗创新应用座谈会\n为深入贯彻落实全省高质量发展大会精神，按照...
2,上海：召开局系统2025年度工作总结表彰大会,https://www.mva.gov.cn/xinwen/dfdt/202603/t202...,2026-03-09,上海：召开局系统2025年度工作总结表彰大会\n近日，上海市退役军人事务局召开局系统2025...
3,湖南：军功耀门楣 “光荣之家”喜上加喜,https://www.mva.gov.cn/xinwen/dfdt/202603/t202...,2026-03-09,湖南：军功耀门楣 “光荣之家”喜上加喜\n军营立功耀家门，喜报千里传美名。近日，湖南多地开展...
4,福建长汀：举行纪念红军长征胜利90周年暨“红土双拥·传承向往”主题教育活动,https://www.mva.gov.cn/xinwen/dfdt/202603/t202...,2026-03-09,福建长汀：举行纪念红军长征胜利90周年暨“红土双拥·传承向往”主题教育活动\n近日，福建省龙...
